# 01 — Signal Characterization Study (Steps 1–10)

**Goal:** Understand what information in rocket telemetry is important before optimizing compression.

Run the full pipeline with `python3 analyze_signal.py` or walk through each step below.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve().parent))
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

from analyze_signal import analyze_signal
from src.characterization.dataset import characterize_all_datasets, write_dataset_report
from src.characterization.frequency_study import analyze_frequency_content
from src.characterization.events import detect_events
from src.importance import compute_importance_mask
from src.characterization.sensitivity import local_compression_sensitivity
from src.characterization.adaptive_experiment import run_adaptive_compression_experiment
from src.data_loader.loader import load_signal
from src.generate_synthetic import create_synthetic_dataset
%matplotlib inline

In [ ]:
# Step 1: Dataset investigation
datasets = characterize_all_datasets('data')
write_dataset_report(datasets, 'reports/dataset_characterization.md')
print('Datasets characterized:', [d.get('name') for d in datasets if 'name' in d])

In [ ]:
# Load primary signal
path = create_synthetic_dataset()
time, signal, meta = load_signal(path)
fs = meta['sampling_frequency']
name = Path(path).stem
print(f'Loaded {name}: {len(signal)} samples at {fs} Hz')

In [ ]:
# Step 2: Frequency content
freq = analyze_frequency_content(signal, fs, name, 'results/frequency_analysis')
print('FFT appropriate:', freq['fft_compression_appropriate'])
print('Top frequency:', freq['dominant_frequencies'][0])
display(Image(filename='results/frequency_analysis/synthetic_rocket_fft_analysis.png', width=700))

In [ ]:
# Steps 3–6: Events, importance, sensitivity, adaptive experiment
events = detect_events(time, signal, fs)
importance, _ = compute_importance_mask(signal)
sensitivity = local_compression_sensitivity(signal, output_dir='results/characterization', name=name)
adaptive = run_adaptive_compression_experiment(signal)
print(f'Events detected: {events["n_events"]}')
print(f'Importance range: {importance.min():.2f} – {importance.max():.2f}')
print(f'Max local MSE: {sensitivity["max_error"]:.4f}')
print(f'Adaptive SNR gain: {adaptive["improvement"]["snr_gain_db"]} dB')

In [ ]:
# Step 9: Full dashboard (also runs noise + failure analysis)
summary = analyze_signal(path)
display(Image(filename=summary['outputs']['dashboard'], width=900))
display(Markdown(open('reports/signal_research_answers.md').read()[:1500]))